# 📘 Використання Azure OpenAI Function Calling

## Вступ

Цей ноутбук демонструє, як використовувати режим **Function Calling** у Microsoft Azure OpenAI з Python SDK `azure.ai.inference`.

Ми розглянемо:
- Що таке Function Calling та його варіанти використання
- Як описати функцію у форматі OpenAI Function Schema
- Як передати її як інструмент (tool) до моделі
- Як обробити відповідь, якщо модель хоче викликати функцію
- Як відповісти від імені функції, щоб модель могла продовжити відповідь
- Як інтегрувати Function Calling в реальний застосунок

> ⚠️ Перед запуском переконайтесь, що у вас є:
> - 🔑 API ключ Azure OpenAI
> - 🌐 Endpoint
> - 📦 Деплоймент моделі GPT (наприклад, `gpt-4`, `gpt-35-turbo`)

## Навчальні цілі

Після завершення цього уроку ви:
- Зрозумієте призначення та переваги Function Calling
- Навчитеся налаштовувати Function Calling у сервісі Azure OpenAI
- Зможете розробляти ефективні функціональні виклики для своїх застосунків
- Зрозумієте повний цикл обробки функціональних викликів

## Розуміння Function Calling

У цьому уроці ми створимо функціонал для освітнього стартапу, який дозволяє користувачам використовувати чат-бота для пошуку технічних курсів. Ми будемо рекомендувати курси, що відповідають їхньому рівню навичок, поточній ролі та технології, яка їх цікавить.

Для цього ми використовуємо комбінацію:
- **Azure OpenAI** для створення інтерактивного чат-досвіду
- **Microsoft Learn Catalog API** для пошуку курсів на основі запиту користувача
- **Function Calling** для обробки запиту користувача та передачі до функції, яка зробить API-запит

### Навіщо використовувати Function Calling?

Якщо ви вже працювали з великими мовними моделями (LLM), ви, напевно, розумієте їх потужність, а також їх обмеження.

Function Calling - це функціональність сервісу Azure OpenAI, яка допомагає подолати такі обмеження:
1. **Послідовний формат відповіді** - отримання структурованих даних
2. **Можливість використовувати дані з інших джерел** у контексті чату

До появи Function Calling відповіді від LLM були неструктурованими та непослідовними. Розробникам доводилося писати складний код валідації, щоб обробляти різні варіанти відповідей.

Користувачі не могли отримати відповіді на запитання типу "Яка зараз погода у Києві?", оскільки моделі обмежені даними, на яких вони навчались.

### Варіанти використання Function Calling

**1. Виклик зовнішніх інструментів**
Чат-боти чудово відповідають на запитання користувачів. Використовуючи Function Calling, чат-боти можуть використовувати повідомлення від користувачів для виконання певних завдань. Наприклад, студент може попросити чат-бота: "Надішли імейл моєму викладачу, що мені потрібна додаткова допомога з цього предмета". Це може викликати функцію `send_email(to: string, body: string)`.

**2. Створення запитів до API або бази даних**
Користувачі можуть знаходити інформацію, використовуючи природну мову, яка перетворюється на форматований запит. Приклад: викладач запитує "Хто з студентів виконав останнє завдання?", що може викликати функцію `get_completed(student_name: string, assignment: int, current_status: string)`.

**3. Створення структурованих даних**
Користувачі можуть взяти блок тексту або CSV і використовувати LLM для вилучення важливої інформації. Наприклад, студент може перетворити статтю з Вікіпедії про мирні угоди для створення AI-карток. Це можна зробити за допомогою функції `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)`.

## 🔐 Крок 1: Підключення до Azure OpenAI

Перший крок – ініціалізувати клієнт `ChatCompletionsClient`, вказавши endpoint та ключ доступу.

In [2]:
import os
import json
import requests
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import ChatCompletionsToolDefinition
from azure.core.credentials import AzureKeyCredential

token = os.environ["GITHUB_TOKEN"]
endpoint = "https://models.inference.ai.azure.com"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

# Виберіть модель загального призначення для тексту
deployment = "gpt-4o"


## 🔧 Крок 2: Визначення функції

Функцію описуємо у форматі JSON Schema (OpenAI Function Calling format) і передаємо як `ChatCompletionsToolDefinition`.

> У нашому випадку, функція `search_courses` — імітує пошук курсів з документації Microsoft Learn за заданими критеріями.

### Елементи функціонального виклику

Ось пояснення ключових частин функціонального виклику:

- **name** - Ім'я функції, яку ми хочемо викликати
- **description** - Опис того, як працює функція. Важливо бути конкретним і чітким
- **parameters** - Список значень та форматів, які модель повинна використовувати у відповіді
- **type** - Тип даних властивостей
- **properties** - Список конкретних значень, які модель буде використовувати для відповіді
- **required** - Обов'язкові властивості для виконання функціонального виклику

In [3]:
# 🔧 Визначення функції, яку модель може викликати
functions = [
    ChatCompletionsToolDefinition(
        type="function",
        function={
            "name": "search_courses",
            "description": "Returns a list of training courses from the Microsoft catalog",
            "parameters": {
                "type": "object",
                "properties": {
                    "role": {
                        "type": "string",
                        "description": """User role (for example: developer, student)"""
                    },
                    "product": {
                        "type": "string",
                        "description": "Covered product (Azure, Power BI, etc.)"
                    },
                    "level": {
                        "type": "string",
                        "description": "User experience level"
                    }
                },
                "required": ["role"]
            }
        }
    )
]


## 📤 Крок 3: Надсилання повідомлення користувача

Ми надсилаємо запит від імені користувача (наприклад, "знайди курси для початківця-розробника по Azure"), і модель вирішує, чи слід викликати функцію.

Щоб інтегрувати функцію у виклик Chat Completion API, ми додаємо `tools=functions` до запиту. Встановлення `tool_choice="auto"` дозволяє LLM самостійно вирішити, яку функцію викликати, виходячи з повідомлення користувача.

In [4]:
# 🧠 Створення повідомлення та виклик моделі з інструментами
messages = [
    {
        "role": "user",
        "content": "Find a course for a beginner developer on Azure"
    }
]

response = client.complete(
    model=deployment,
    messages=messages,
    tools=functions,
    tool_choice="auto"
)

response_message = response.choices[0].message
print("📥 Відповідь моделі:")
print(response_message)


📥 Відповідь моделі:
{'annotations': [], 'content': None, 'refusal': None, 'role': 'assistant', 'tool_calls': [{'function': {'arguments': '{"role":"developer","product":"Azure","level":"beginner"}', 'name': 'search_courses'}, 'id': 'call_mXmnh1pZ3amNKeHuxR3E3iC3', 'type': 'function'}]}


## ⚙️ Крок 4: Обробка виклику функції (tool_call)

Якщо модель вирішила викликати функцію, ми зчитуємо її аргументи, викликаємо локальну функцію `search_courses`, і передаємо результат назад в модель як `"role": "tool"`.

Після цього модель формує підсумкову відповідь з урахуванням виклику.

### Інтеграція у застосунок

Для інтеграції у реальний застосунок необхідно:
1. Перевірити, чи модель хоче викликати функцію
2. Отримати аргументи з відповіді моделі
3. Викликати відповідну функцію з отриманими аргументами
4. Додати відповідь від функції до історії повідомлень
5. Зробити новий запит до моделі для отримання підсумкової відповіді користувачу

In [5]:
# ⚙️ Обробка function_call (tool_call)
tool_calls = response_message.tool_calls

if tool_calls and len(tool_calls) > 0:
    first_tool_call = tool_calls[0]
    function_name = first_tool_call.function.name
    function_args = json.loads(first_tool_call.function.arguments)

    def search_courses(role, product, level):
        url = "https://learn.microsoft.com/api/catalog/"
        params = {
            "role": role,
            "product": product,
            "level": level
        }
        response = requests.get(url, params=params)
        modules = response.json().get("modules", [])
        results = []
        for module in modules[:5]:
            title = module.get("title")
            url = module.get("url")
            results.append({"title": title, "url": url})
        return json.dumps(results, ensure_ascii=False)
        

    available_functions = {
        "search_courses": search_courses,
    }

    function_to_call = available_functions[function_name]
    function_response = function_to_call(**function_args)

    print("✅ Результат виклику функції:")
    print(function_response)

    # Додаємо відповідь моделі з tool_calls до історії повідомлень
    messages.append({
        "role": "assistant",
        "content": "Відповідай українською",
        "tool_calls": [
            {
                "id": first_tool_call.id,
                "type": "function",
                "function": {
                    "name": first_tool_call.function.name,
                    "arguments": first_tool_call.function.arguments
                }
            }
        ]
    })

    # Додаємо відповідь функції
    messages.append({
        "role": "tool",  # У найновіших версіях API використовується "tool" замість "function"
        "tool_call_id": first_tool_call.id,  # Це критично!
        "name": function_name,
        "content": function_response
    })

    # Отримуємо фінальну відповідь від моделі
    next_response = client.complete(
        model=deployment,
        messages=messages
    )

    print("📤 Остаточна відповідь моделі:")
    print(next_response.choices[0].message.content)
else:
    print("⚠️ Модель не захотіла викликати функцію.")


✅ Результат виклику функції:
[{"title": "Guide AI workload operations with an AI Center of Excellence", "url": "https://learn.microsoft.com/en-us/training/modules/guide-ai-operations-center-excellence/?WT.mc_id=api_CatalogApi"}, {"title": "Develop products with screen reader support", "url": "https://learn.microsoft.com/en-us/training/modules/develop-products-with-screen-reader-support/?WT.mc_id=api_CatalogApi"}, {"title": "Host a web application with Azure App Service", "url": "https://learn.microsoft.com/en-us/training/modules/host-a-web-app-with-azure-app-service/?WT.mc_id=api_CatalogApi"}, {"title": "Deploy to multiple Azure environments by using JSON ARM template features", "url": "https://learn.microsoft.com/en-us/training/modules/modify-azure-resource-manager-template-reuse/?WT.mc_id=api_CatalogApi"}, {"title": "Secure Azure OpenAI authentication and authorization", "url": "https://learn.microsoft.com/en-us/training/modules/secure-azure-openai-authentication-authorization/?WT.mc

## Висновки та практичні завдання

Тепер ви знаєте, як інтегрувати Function Calling у ваші застосунки з Azure OpenAI. Ця потужна функціональність дозволяє створювати більш інтелектуальні та корисні взаємодії з користувачами.

### Завдання для подальшого вивчення:

1. Додайте більше параметрів до функції, які можуть допомогти учням знаходити більше курсів. Перегляньте доступні API параметри [тут](https://learn.microsoft.com/training/support/catalog-api-developer-reference).
2. Створіть інший функціональний виклик, який отримує додаткову інформацію від користувача, наприклад, його рідну мову.
3. Додайте обробку помилок для випадків, коли функціональний виклик та/або API-виклик не повертає відповідних курсів.
4. Розширте функціонал для рекомендації не лише курсів, але й навчальних шляхів (learning paths).

## Висновки та практичні завдання

Тепер ви знаєте, як інтегрувати Function Calling у ваші застосунки з Azure OpenAI. Ця потужна функціональність дозволяє створювати більш інтелектуальні та корисні взаємодії з користувачами.

### Завдання для подальшого вивчення:

1. Додайте більше параметрів до функції, які можуть допомогти учням знаходити більше курсів. Перегляньте доступні API параметри [тут](https://learn.microsoft.com/training/support/catalog-api-developer-reference).
2. Створіть інший функціональний виклик, який отримує додаткову інформацію від користувача, наприклад, його рідну мову.
3. Додайте обробку помилок для випадків, коли функціональний виклик та/або API-виклик не повертає відповідних курсів.
4. Розширте функціонал для рекомендації не лише курсів, але й навчальних шляхів (learning paths).

### Індивідуальне завдання

> Варіант 9 - containers

**Частина 1**: Модифікуйте базовий ноутбук, щоб реалізувати пошук курсів за іншою тематикою (згідно з вашим варіантом)

**Частина 2**: Розширте функціональність базового рішення одним із наступних способів: 

In [ ]:
import os
import json
import requests
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import ChatCompletionsToolDefinition
from azure.core.credentials import AzureKeyCredential
import traceback

# 🔐 Токен GitHub Models
token = os.environ["GITHUB_TOKEN"]
endpoint = "https://models.inference.ai.azure.com"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

deployment = "gpt-4o"

# ============================
# 🛠 ОПИС ІНСТРУМЕНТІВ (TOOLS)
# ============================

functions = [
    ChatCompletionsToolDefinition(
        type="function",
        function={
            "name": "search_courses",
            "description": (
                "Search Microsoft Learn catalog for training about containers "
                "(Docker, Kubernetes, контейнеризація). Підтримує розширену фільтрацію."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "role": {
                        "type": "string",
                        "description": "User role (e.g. developer, devops, student)"
                    },
                    "subject": {
                        "type": "string",
                        "description": (
                            "Covered subject. Для варіанту 9 зазвичай це 'containers', "
                            "'docker', 'kubernetes' тощо, зчитується з промпту користувача."
                        )
                    },
                    "level": {
                        "type": "string",
                        "description": "User experience level: beginner, intermediate, advanced"
                    },
                    "content_type": {
                        "type": "string",
                        "description": "Type of content: 'module', 'learningPath' тощо"
                    },
                    "locale": {
                        "type": "string",
                        "description": "Locale of the content, e.g. 'en-us', 'uk-ua'"
                    },
                    "max_duration": {
                        "type": "integer",
                        "description": "Maximum duration in minutes (e.g. 240)"
                    },
                    "min_rating": {
                        "type": "number",
                        "description": "Minimal average rating (e.g. 4.5)"
                    }
                },
                "required": ["role"]
            }
        }
    ),
    ChatCompletionsToolDefinition(
        type="function",
        function={
            "name": "course_info",
            "description": "Return detailed info on a specific course using its UID",
            "parameters": {
                "type": "object",
                "properties": {
                    "course_id": {
                        "type": "string",
                        "description": "The unique UID of the course (module or learning path)"
                    }
                },
                "required": ["course_id"]
            }
        }
    )
]

# ============================
# 🧩 РЕАЛІЗАЦІЯ ФУНКЦІЙ
# ============================

def search_courses(role=None,
                   subject=None,
                   level=None,
                   content_type=None,
                   locale=None,
                   max_duration=None,
                   min_rating=None):
    """
    Розширений пошук курсів Microsoft Learn.
    Використовується як tool для варіанту 9 (containers).
    """
    print("[search_courses] "
          f"role={role}, subject={subject}, level={level}, "
          f"type={content_type}, locale={locale}, "
          f"max_duration={max_duration}, min_rating={min_rating}")

    url = "https://learn.microsoft.com/api/catalog/"

    # Параметри запиту до API
    params = {}
    if role:
        params["role"] = role
    if subject:
        params["subject"] = subject
    if level:
        params["level"] = level
    if content_type:
        params["type"] = content_type
    if locale:
        params["locale"] = locale

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    # Беремо як окремі модулі, так і навчальні шляхи
    courses = data.get("modules", []) + data.get("learningPaths", [])

    # Перетворюємо пороги, якщо вони прийшли як рядки
    if max_duration is not None:
        try:
            max_duration = int(max_duration)
        except (TypeError, ValueError):
            max_duration = None

    if min_rating is not None:
        try:
            min_rating = float(min_rating)
        except (TypeError, ValueError):
            min_rating = None

    results = []
    for course in courses:
        rating = None
        rating_obj = course.get("rating") or {}
        if isinstance(rating_obj, dict):
            rating = rating_obj.get("average")

        duration = course.get("duration_in_minutes")

        # Фільтрація за рейтингом
        if min_rating is not None and rating is not None and rating < min_rating:
            continue

        # Фільтрація за тривалістю
        if max_duration is not None and duration is not None and duration > max_duration:
            continue

        results.append({
            "title": course.get("title"),
            "uid": course.get("uid"),
            "rating": rating,
            "duration": duration,
            "type": course.get("type")
        })

    # Сортуємо: спочатку за рейтингом (спадання), при бажанні можна додати додаткові ключі
    results.sort(
        key=lambda x: (x["rating"] if x["rating"] is not None else 0.0),
        reverse=True
    )

    # Повертаємо топ-5
    return json.dumps(results[:5], ensure_ascii=False)


def course_info(course_id):
    """
    Детальна інформація про курс/модуль за UID.
    """
    print(f"[course_info] Getting details for UID: {course_id}")
    url = f"https://learn.microsoft.com/api/catalog/?uid={course_id}"
    response = requests.get(url)
    if response.status_code != 200:
        return json.dumps({"error": "Course not found"}, ensure_ascii=False)

    data = response.json()
    course_list = data.get("courses") or data.get("modules") or data.get("learningPaths")
    if not course_list:
        return json.dumps({"error": "Parse error"}, ensure_ascii=False)

    course = course_list[0]

    rating = None
    rating_obj = course.get("rating") or {}
    if isinstance(rating_obj, dict):
        rating = rating_obj.get("average")

    details = {
        "title": course.get("title"),
        "summary": course.get("summary"),
        "duration": course.get("duration_in_minutes"),
        "rating": rating,
        "level": course.get("level"),
        "locale": course.get("locale"),
        "url": "https://learn.microsoft.com" + course.get("url", "")
    }
    return json.dumps(details, ensure_ascii=False)


available_functions = {
    "search_courses": search_courses,
    "course_info": course_info
}

# ============================
# 💬 ПОЧАТКОВІ ПОВІДОМЛЕННЯ
# (ВАРІАНТ 9: CONTAINERS)
# ============================

messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant. Always reply in Ukrainian."
    },
    {
        "role": "system",
        "content": (
            "Ти допомагаєш користувачу знаходити курси Microsoft Learn "
            "на тему контейнеризації (containers, Docker, Kubernetes). "
            "Доступні значення для level: beginner, intermediate, advanced. "
            "Значення subject ти повинен діставати з промпту користувача. "
            "Якщо користувач не вказав locale, за замовчуванням використовуй 'en-us'."
        )
    },
    {
        "role": "user",
        "content": (
            "Debug mode active. Я початківець-розробник (beginner developer) "
            "і хочу навчитися працювати з containers (Docker, Kubernetes). "
            "Знайди найкращі beginner-курси про containers на Microsoft Learn "
            "з рейтингом не нижче 4.5 і тривалістю не більше 240 хвилин. "
            "Потім обери той, що має найвищий рейтинг, і отримай повну "
            "інформацію про нього. Для всіх звернень до каталогу використовуй доступні функції."
        )
    }
]

# ============================
# 🔄 ЦИКЛ ВИКЛИКУ TOOLS
# ============================

try:
    response = client.complete(
        model=deployment,
        messages=messages,
        tools=functions,
        tool_choice="auto"
    )

    # LOOP: поки модель хоче викликати інструменти
    while response.choices[0].message.tool_calls:
        tool_calls = response.choices[0].message.tool_calls

        # Додаємо запит моделі до історії
        messages.append(response.choices[0].message)

        for tool_call in tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)

            if function_name in available_functions:
                function_to_call = available_functions[function_name]

                # Виконуємо локальну Python-функцію
                tool_result = function_to_call(**function_args)

                # Додаємо відповідь інструмента в історію
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": tool_result
                })

        # Другий (та наступні) запити до моделі з оновленою історією
        response = client.complete(
            model=deployment,
            messages=messages,
            tools=functions,
            tool_choice="auto"
        )

    # 🧾 Фінальна відповідь моделі
    print("Final Answer:")
    print(response.choices[0].message.content)

except Exception as e:
    print("Сталася помилка під час виконання:")
    traceback.print_exc()


[search_courses] role=developer, subject=containers, level=beginner, type=None, locale=en-us, max_duration=240, min_rating=4.5
[course_info] Getting details for UID: learn.languages.dotnet-deploy-microservices-kubernetes
Final Answer:
Найкращий курс для початківців-розробників на тему контейнерів - це "Deploy a .NET microservice to Kubernetes". 

### Інформація про курс:
- **Назва:** Deploy a .NET microservice to Kubernetes
- **Опис:** Дізнайтесь, як розгорнути мікросервісні додатки у контейнерах за допомогою Kubernetes. Курс також охоплює виклики управління та моніторингу контейнерів у складних рішеннях, та як Kubernetes допомагає вирішувати такі задачі.
- **Рейтинг:** 4.81
- **Тривалість:** 26 хвилин
- **Мова:** англійська (en-us)
- **Посилання:** [Відкрити курс](https://learn.microsoft.comhttps://learn.microsoft.com/en-us/training/modules/dotnet-deploy-microservices-kubernetes/?WT.mc_id=api_CatalogApi)

Це короткий й дуже високо оцінений курс, який дозволить вам розпочати практику з